## Step 1 Data preparing
I simulated data for two quarters.
**data_last** represents the previous quarter, which is used for model training.
**data_now** represents the current quarter, where the trained models are applied to automatically generate a simplified quarterly report.

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(123)

n_users = 2000

data_last = pd.DataFrame({
    "client_id": np.random.randint(1000, 1100, n_users),
    "month": np.random.choice(["Mar", "Apr", "May"], n_users),
    "age": np.random.randint(21, 65, n_users),
    "income": np.random.normal(6000, 2000, n_users).clip(1000, 20000),
    "gender": np.random.choice(["M", "F"], n_users),
    "num_inquiries": np.random.poisson(3, n_users),
    "num_transactions": np.random.poisson(10, n_users),
    "last_purchase": np.random.normal(400, 400, n_users).clip(0, 1000),
    "avg_transaction_value": np.random.normal(60, 20, n_users).clip(20, 100),
    "credit_score": np.random.normal(650, 70, n_users).clip(300, 850),
    "utilization": np.random.uniform(0.1, 1.0, n_users),
    "loan_amount": np.random.normal(5000, 1500, n_users).clip(500, 10000),
    "repayment": np.random.normal(4000, 1200, n_users).clip(300, 9000),
    "status": np.random.choice(["approved", "rejected", "pending"], n_users)
})

data_last.head()

,client_id,month,age,income,gender,num_inquiries,num_transactions,last_purchase,avg_transaction_value,credit_score,utilization,loan_amount,repayment,status
0,1066,May,30,6978.121810,F,3,6,420.789587,92.150426,738.646962,0.412172,5999.170312,5613.745617,pending
1,1092,Mar,53,5750.258815,F,6,6,564.474710,75.313303,589.242729,0.799150,6365.231485,4326.376467,approved
2,1098,Mar,56,8773.436814,F,2,9,0.000000,93.192645,499.529512,0.969503,4684.361006,4757.940827,approved
3,1017,May,31,5744.520572,M,2,7,0.000000,44.628735,721.885285,0.650373,5600.101572,6773.426744,rejected
4,1083,May,53,7204.373847,M,1,12,268.451702,58.981192,650.296389,0.853064,7763.540791,3262.387827,rejected


In [38]:
data_prev = data_last.groupby("client_id").agg({
    "avg_transaction_value": "mean",
    "num_transactions": "mean"
}).reset_index()

In [39]:
np.random.seed(321)

n_users = 2000

data_now = pd.DataFrame({
    "client_id": np.random.randint(1000, 1100, n_users),
    "month": np.random.choice(["Jun", "Jul", "Aug"], n_users),
    "age": np.random.randint(21, 65, n_users),
    "income": np.random.normal(6000, 2000, n_users).clip(1000, 20000),
    "gender": np.random.choice(["M", "F"], n_users),
    "num_inquiries": np.random.poisson(3, n_users),
    "num_transactions": np.random.poisson(10, n_users),

    "last_purchase": np.nan,  # 先空着

    "avg_transaction_value": np.random.normal(200, 80, n_users).clip(20, 1000),
    "credit_score": np.random.normal(650, 70, n_users).clip(300, 850),
    "utilization": np.random.uniform(0.1, 1.0, n_users),
    "loan_amount": np.random.normal(5000, 1500, n_users).clip(500, 10000),
    "repayment": np.random.normal(4000, 1200, n_users).clip(300, 9000),
    "status": np.random.choice(["approved", "rejected", "pending"], n_users)
})
data_now = data_now.merge(
    data_prev,
    on="client_id",
    how="left",
    suffixes=("", "_prev")
)
data_now["avg_transaction_value_prev"] = data_now["avg_transaction_value_prev"].fillna(0)
data_now["num_transactions_prev"] = data_now["num_transactions_prev"].fillna(0)
data_now["last_purchase"] = (
    data_now["avg_transaction_value_prev"] *
    data_now["num_transactions_prev"]
)
data_now = data_now.drop(columns=[
    "avg_transaction_value_prev",
    "num_transactions_prev"
])
data_now.head()

,client_id,month,age,income,gender,num_inquiries,num_transactions,last_purchase,avg_transaction_value,credit_score,utilization,loan_amount,repayment,status
0,1026,Jun,34,4566.429792,F,1,7,652.883461,354.774663,614.891873,0.547388,5517.706940,5421.730000,approved
1,1031,Aug,48,8892.662671,F,1,8,540.278188,124.331871,512.476509,0.227387,6928.484060,3234.657231,approved
2,1041,Aug,42,6188.271642,F,2,11,650.106615,244.567755,694.026883,0.961184,2710.371639,2446.170515,rejected
3,1072,Jun,63,5127.897877,M,4,7,509.819450,162.934567,818.671568,0.365212,5184.515551,4678.760162,rejected
4,1017,Jun,32,5931.589066,M,3,6,512.276051,202.576134,643.490503,0.667985,6212.818893,5341.107783,pending


## Step 2 Modeling
### 1） Risk Churn
A composite risk score was constructed using a weighted combination of credit score, utilization, and repayment behavior.
Customers were then segmented into three risk categories (Low, Medium, High) based on the distribution of the risk score.
Finally, the categorical risk levels were encoded into ordered numerical values to enable modeling using an ordered logistic regression framework.

In [41]:
data_last["risk_score"] = (
    (1 - data_last["credit_score"] / 850) * 0.5 +
    data_last["utilization"] * 0.3 +
    (data_last["loan_amount"] - data_last["repayment"]) / data_last["loan_amount"] * 0.2
)

data_last["risk_level"] = pd.qcut(
    data_last["risk_score"],
    3,
    labels=["Low", "Medium", "High"]
)

In [48]:
data_last["risk_level_num"] = data_last["risk_level"].map({
    "Low": 0,
    "Medium": 1,
    "High": 2
})

In [49]:
from statsmodels.miscmodels.ordinal_model import OrderedModel

X = data_last[[
    "income",
    "credit_score",
    "utilization",
    "loan_amount",
    "repayment"
]]

y = data_last["risk_level_num"]

model_risk = OrderedModel(
    y,
    X,
    distr='logit'
)

res_risk = model_risk.fit(method='bfgs')

print(res_risk.summary())

Optimization terminated successfully.
         Current function value: 0.225561
         Iterations: 51
         Function evaluations: 58
         Gradient evaluations: 58
                             OrderedModel Results                             
Dep. Variable:         risk_level_num   Log-Likelihood:                -451.12
Model:                   OrderedModel   AIC:                             916.2
Method:            Maximum Likelihood   BIC:                             955.5
Date:                Mon, 23 Mar 2026                                         
Time:                        00:50:09                                         
No. Observations:                2000                                         
Df Residuals:                    1993                                         
Df Model:                           5                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------

In [46]:
data_last["spending_ratio"] = (
    data_last["current_spending"] /
    (data_last["last_purchase"] + 1)
)

data_last["churn_flag"] = (data_last["spending_ratio"] < 1).astype(int)

In [51]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X = data_last[[
    "last_purchase",
    "num_transactions",
    "avg_transaction_value",
    "credit_score",
    "income"
]]

y = data_last["churn_flag"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

model_churn = LogisticRegression(max_iter=1000)
model_churn.fit(X_train, y_train)

print("Accuracy:", model_churn.score(X_test, y_test))

Accuracy: 0.9766666666666667


### 2）Customer Churn
Customer churn is defined based on behavioral indicators, particularly recency and spending activity.
Customers with a long time since their last purchase (high recency) and low current spending levels are more likely to churn.
Initially, a rule-based approach was used to generate churn labels based on these criteria.
A logistic regression model was then trained to learn the relationship between customer characteristics and churn behavior.
The trained model was applied to new data to predict churn probability, enabling proactive identification of at-risk customers.

In [52]:
X_new = data_now[[
    "income",
    "credit_score",
    "utilization",
    "loan_amount",
    "repayment"
]]

In [55]:
pred_probs = res_risk.model.predict(
    res_risk.params,
    exog=X_new
)

In [58]:
data_now["risk_level_pred_num"] = np.argmax(pred_probs, axis=1)

In [59]:
pred_probs = res_risk.model.predict(
    res_risk.params,
    exog=data_now[[
        "income",
        "credit_score",
        "utilization",
        "loan_amount",
        "repayment"
    ]]
)

data_now["risk_level_pred_num"] = np.argmax(pred_probs, axis=1)

mapping = {0: "Low", 1: "Medium", 2: "High"}

data_now["risk_level_pred"] = data_now["risk_level_pred_num"].map(mapping)

In [60]:
data_now["risk_level_pred"].value_counts()

risk_level_pred
Low       706
Medium    659
High      635
Name: count, dtype: int64

In [61]:
data_now["risk_level_pred"].value_counts()

risk_level_pred
Low       706
Medium    659
High      635
Name: count, dtype: int64

In [64]:
data_now.groupby("risk_level_pred")[[
    "income",
    "credit_score",
    "utilization"
]].mean()

,income,credit_score,utilization
risk_level_pred,,,
High,6032.348627,624.352874,0.758716
Low,6100.788936,673.392132,0.341385
Medium,6027.274514,646.780719,0.539361


In [66]:
X_new = data_now[[
    "last_purchase",
    "num_transactions",
    "avg_transaction_value",
    "credit_score",
    "income"
]]

In [67]:
data_now["churn_pred"] = model_churn.predict(X_new)

In [68]:
data_now["churn_prob"] = model_churn.predict_proba(X_new)[:, 1]

In [69]:
data_now["churn_pred"].value_counts()

churn_pred
0    1897
1     103
Name: count, dtype: int64

In [70]:
data_now["churn_pred"].value_counts(normalize=True)

churn_pred
0    0.9485
1    0.0515
Name: proportion, dtype: float64

In [83]:
high_churn = (data_now["churn_prob"] > 0.7).sum()

## Step 3 Automatically Report
To improve code structure and reusability, the reporting logic was encapsulated into a function, following basic principles of object-oriented programming (OOP) such as modularity and abstraction.
Instead of writing repetitive code, the function takes a dataset as input and automatically generates a simplified quarterly report, including key metrics such as total customers, risk distribution, and churn rate.
This approach makes the solution scalable and easy

In [84]:
def run_quarterly_report(data):
    import pandas as pd

    total_customers = len(data)
    high_risk_pct = (data["risk_level_pred"] == "High").mean()
    churn_rate = data["churn_pred"].mean()

    high_churn = (data["churn_prob"] > 0.7).sum() if "churn_prob" in data.columns else None

    risk_dist = data["risk_level_pred"].value_counts()
    churn_dist = data["churn_pred"].value_counts()

    summary = f"""
==============================
Quarterly Report Summary
==============================

Total customers: {total_customers}

High risk customers: {high_risk_pct*100:.1f}%
Churn rate: {churn_rate*100:.1f}%
"""

    if high_churn is not None:
        summary += f"\nHigh churn probability customers: {high_churn}\n"

    print(summary)

    summary_df = pd.DataFrame({
        "Metric": ["Total Customers", "High Risk %", "Churn Rate"],
        "Value": [total_customers, high_risk_pct, churn_rate]
    })

    print("\nSummary Table:")
    print(summary_df)

    print("\nRisk Distribution:")
    print(risk_dist)

    print("\nChurn Distribution:")
    print(churn_dist)

    return summary_df

In [85]:
run_quarterly_report(data_now)


Quarterly Report Summary

Total customers: 2000

High risk customers: 31.8%
Churn rate: 5.1%

High churn probability customers: 95


Summary Table:
            Metric      Value
0  Total Customers  2000.0000
1      High Risk %     0.3175
2       Churn Rate     0.0515

Risk Distribution:
risk_level_pred
Low       706
Medium    659
High      635
Name: count, dtype: int64

Churn Distribution:
churn_pred
0    1897
1     103
Name: count, dtype: int64


,Metric,Value
0,Total Customers,2000.0000
1,High Risk %,0.3175
2,Churn Rate,0.0515


## Business Insights

- High-risk customers tend to have lower credit scores and higher utilization rates, indicating potential repayment issues.
- Customers with high churn probability show low recent activity and low spending levels.
- High-potential customers with low risk represent valuable targets for marketing and retention strategies.

Overall, the automated system enables the company to identify key customer segments and support data-driven decision-making.